# Toronto Zoo Visitor Experience: SQL Exploratory Analysis

## Load Data into SQLite

In [95]:
import pandas as pd
import sqlite3

df = pd.read_csv('Survey_2024_Themed_Cleaned.csv')
conn = sqlite3.connect('EDA_Summary_SQL.db')
df.to_sql('survey', conn, if_exists='replace', index=False)

def sql(query):
    return pd.read_sql(query, conn)

print(f'Loaded {len(df)} rows, {len(df.columns)} columns into EDA_Summary_SQL.db')

Loaded 7522 rows, 126 columns into EDA_Summary_SQL.db


## 1. Data Profiling
How large is the dataset? What's the response rate for key fields? What date range does it cover?

In [96]:
sql("""
SELECT 
    COUNT(*) AS total_responses,
    MIN(Date) AS earliest_response,
    MAX(Date) AS latest_response,
    ROUND(100.0 * SUM(CASE WHEN Overall_Sat IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_rated_overall,
    ROUND(100.0 * SUM(CASE WHEN Likely_Rec IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_rated_recommend,
    ROUND(100.0 * SUM(Membership) / COUNT(*), 1) AS pct_members,
    ROUND(100.0 * SUM(Local) / COUNT(*), 1) AS pct_local,
    ROUND(100.0 * SUM(left_any_comment) / COUNT(*), 1) AS pct_left_comment
FROM survey
""")

,total_responses,earliest_response,latest_response,pct_rated_overall,pct_rated_recommend,pct_members,pct_local,pct_left_comment
0,7522,2024-01-04,2024-12-31,100.0,100.0,42.4,66.5,64.7


## 2. CSAT and NPS
What percentage of visitors are satisfied (rating >= 4)?<br>
What percentage are likely to recommend (rating >= 8)?

In [97]:
sql("""
SELECT
    ROUND(100.0 * SUM(CASE WHEN Overall_Sat >= 4 THEN 1 ELSE 0 END) 
        / COUNT(*), 2) AS csat_pct,
    ROUND(AVG(Overall_Sat), 2) AS mean_satisfaction,
    COUNT(*) AS n_rated
FROM survey
WHERE Overall_Sat IS NOT NULL
""")

,csat_pct,mean_satisfaction,n_rated
0,74.22,3.94,7522


In [98]:
sql("""
WITH nps_segments AS (
    SELECT
        CASE 
            WHEN Likely_Rec <= 6 THEN 'Detractor'
            WHEN Likely_Rec <= 8 THEN 'Passive'
            ELSE 'Promoter'
        END AS segment,
        COUNT(*) AS n
    FROM survey
    WHERE Likely_Rec IS NOT NULL
    GROUP BY segment
)
SELECT 
    segment,
    n,
    ROUND(100.0 * n / SUM(n) OVER(), 1) AS pct
FROM nps_segments
ORDER BY 
    CASE segment 
        WHEN 'Detractor' THEN 1 
        WHEN 'Passive' THEN 2 
        ELSE 3 
    END
""")

,segment,n,pct
0,Detractor,1784,23.7
1,Passive,1977,26.3
2,Promoter,3761,50.0


## 3. Membership and Local Visitor Comparisons
Do members and local visitors differ in satisfaction and recommendation likelihood?

In [99]:
sql("""
SELECT
    CASE WHEN Membership = 1 THEN 'Member' ELSE 'Non-Member' END AS group_name,
    COUNT(*) AS n,
    ROUND(AVG(Overall_Sat), 2) AS avg_satisfaction,
    ROUND(AVG(Likely_Rec), 2) AS avg_recommend,
    ROUND(100.0 * SUM(CASE WHEN Overall_Sat >= 4 THEN 1 ELSE 0 END) 
        / NULLIF(SUM(CASE WHEN Overall_Sat IS NOT NULL THEN 1 ELSE 0 END), 0), 1) AS pct_satisfied,
    ROUND(100.0 * SUM(CASE WHEN Likely_Rec >= 9 THEN 1 ELSE 0 END) 
        / NULLIF(SUM(CASE WHEN Likely_Rec IS NOT NULL THEN 1 ELSE 0 END), 0), 1) AS pct_promoter
FROM survey
GROUP BY Membership

UNION ALL

SELECT
    CASE WHEN Local = 1 THEN 'Local' ELSE 'Non-Local' END,
    COUNT(*),
    ROUND(AVG(Overall_Sat), 2),
    ROUND(AVG(Likely_Rec), 2),
    ROUND(100.0 * SUM(CASE WHEN Overall_Sat >= 4 THEN 1 ELSE 0 END) 
        / NULLIF(SUM(CASE WHEN Overall_Sat IS NOT NULL THEN 1 ELSE 0 END), 0), 1),
    ROUND(100.0 * SUM(CASE WHEN Likely_Rec >= 9 THEN 1 ELSE 0 END) 
        / NULLIF(SUM(CASE WHEN Likely_Rec IS NOT NULL THEN 1 ELSE 0 END), 0), 1)
FROM survey
GROUP BY Local
""")

,group_name,n,avg_satisfaction,avg_recommend,pct_satisfied,pct_promoter
0,Non-Member,4333,3.75,7.11,67.7,40.5
1,Member,3189,4.19,8.59,83.1,63.0
2,Non-Local,2522,3.86,7.33,71.0,43.5
3,Local,5000,3.98,7.95,75.9,53.3


## 4. Conservation Learning Impact
Did visitors report learning more about conservation after their visit? Does this relate to satisfaction?

In [100]:
sql("""
WITH learning AS (
    SELECT
        CASE WHEN Con_After > Con_Before THEN 'Learned More'
             WHEN Con_After = Con_Before THEN 'No Change'
             ELSE 'Learned Less'
        END AS learning_outcome,
        Overall_Sat,
        Likely_Rec
    FROM survey
    WHERE Con_Before IS NOT NULL AND Con_After IS NOT NULL
)
SELECT
    learning_outcome,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS pct,
    ROUND(AVG(Overall_Sat), 2) AS avg_satisfaction,
    ROUND(AVG(Likely_Rec), 2) AS avg_recommend
FROM learning
GROUP BY learning_outcome
ORDER BY avg_satisfaction DESC
""")

,learning_outcome,n,pct,avg_satisfaction,avg_recommend
0,Learned More,1272,16.9,4.23,8.47
1,No Change,5895,78.4,3.90,7.63
2,Learned Less,355,4.7,3.59,6.87


## 5. Monthly Trends
Does satisfaction vary by month of visit?

In [101]:
sql("""
SELECT
    Month_Visited,
    COUNT(*) AS n_responses,
    ROUND(AVG(Overall_Sat), 2) AS avg_satisfaction,
    ROUND(AVG(Likely_Rec), 2) AS avg_recommend,
    ROUND(100.0 * SUM(left_any_comment) / COUNT(*), 1) AS pct_left_comment,
    ROUND(AVG(Overall_Sat) - LAG(AVG(Overall_Sat)) OVER (ORDER BY Month_Visited), 2) AS sat_change_vs_prev_month
FROM survey
GROUP BY Month_Visited
ORDER BY Month_Visited
""")

,Month_Visited,n_responses,avg_satisfaction,avg_recommend,pct_left_comment,sat_change_vs_prev_month
0,1,8,4.50,9.63,62.5,NaN
1,2,35,4.11,8.31,74.3,-0.39
2,3,754,3.90,7.83,71.4,-0.22
3,4,631,3.98,8.04,66.9,0.09
4,5,807,3.79,7.34,65.3,-0.19
5,6,793,3.79,7.38,65.3,-0.00
6,7,1263,3.77,7.28,64.1,-0.02
7,8,1583,4.04,7.86,63.9,0.27
8,9,786,4.07,8.05,62.0,0.03
9,10,462,4.03,7.94,59.7,-0.05


## 6. Sub-Area Satisfaction and Attendance
Which sub-areas are most visited? Which are most and least liked? This maps to the quadrant analysis in the main pipeline.

In [102]:
sql("""
WITH sub_areas AS (
    SELECT 'African Rainforest' AS area, Area_Sat_African_Rainforest_Pavilion AS sat FROM survey
    UNION ALL SELECT 'African Savannah', Area_Sat_African_Savannah FROM survey
    UNION ALL SELECT 'Australasia Pavilion', Area_Sat_Australasia_Pavilion FROM survey
    UNION ALL SELECT 'Americas Pavilion', Area_Sat_Americas_Pavilion FROM survey
    UNION ALL SELECT 'Canadian Domain', Area_Sat_Canadian_Domain FROM survey
    UNION ALL SELECT 'Eurasian Wilds', Area_Sat_Eurasian_Wilds FROM survey
    UNION ALL SELECT 'Indo Malaya', Area_Sat_Indo_Malaya_Pavilion FROM survey
    UNION ALL SELECT 'Tundra Trek', Area_Sat_Tundra_Trek FROM survey
    UNION ALL SELECT 'Mayan Temple', Area_Sat_Mayan_Temple FROM survey
    UNION ALL SELECT 'Kids Zoo', Area_Sat_Kids_Zoo FROM survey
    UNION ALL SELECT 'Greenhouse', Area_Sat_Greenhouse FROM survey
    UNION ALL SELECT 'Wildlife Health Centre', Area_Sat_Wildlife_Health_Centre FROM survey
    UNION ALL SELECT 'Splash Island', Area_Sat_Splash_Island FROM survey
),
agg AS (
    SELECT 
        area,
        COUNT(sat) AS n_rated,
        ROUND(100 * COUNT(sat)/7522, 1) AS pct_attended,
        ROUND(AVG(sat), 2) AS avg_satisfaction,
        ROUND(100 * SUM(CASE WHEN sat >= 4 THEN 1 ELSE 0 END) / COUNT(sat), 1) AS pct_satisfied
    FROM sub_areas
    WHERE sat IS NOT NULL
    GROUP BY area
)
SELECT
    a.*,
    CASE 
        WHEN pct_attended > (SELECT AVG(pct_attended) FROM agg)
         AND pct_satisfied < (SELECT AVG(pct_satisfied) FROM agg)
        THEN 'PAIN POINT'
        WHEN pct_attended <= (SELECT AVG(pct_attended) FROM agg)
         AND pct_satisfied >= (SELECT AVG(pct_satisfied) FROM agg)
        THEN 'Hidden Gem'
        ELSE 'Monitor'
    END AS quadrant,
    RANK() OVER (ORDER BY pct_satisfied) AS sat_rank
FROM agg a
ORDER BY sat_rank
""")

,area,n_rated,pct_attended,avg_satisfaction,pct_satisfied,quadrant,sat_rank
0,Canadian Domain,2227,29.0,3.12,41.0,Monitor,1
1,Kids Zoo,1677,22.0,3.44,52.0,Monitor,2
2,Mayan Temple,2939,39.0,3.67,59.0,Monitor,3
3,Eurasian Wilds,5297,70.0,3.76,63.0,PAIN POINT,4
4,Wildlife Health Centre,1751,23.0,3.89,66.0,Hidden Gem,5
5,Americas Pavilion,4636,61.0,3.87,67.0,Monitor,6
6,Australasia Pavilion,5059,67.0,3.87,67.0,Monitor,6
7,Indo Malaya,5101,67.0,3.90,68.0,Monitor,8
8,Splash Island,1081,14.0,3.83,68.0,Hidden Gem,8
9,Tundra Trek,5874,78.0,3.88,68.0,Monitor,8


## 7. Global Satisfaction Features
How do visitors rate specific aspects of the zoo experience (food, washrooms, signage, etc.)?

In [103]:
sql("""
WITH features AS (
    SELECT 'Food Services' AS feature, Sat_Food AS sat FROM survey
    UNION ALL SELECT 'Washrooms', Sat_Washrooms FROM survey
    UNION ALL SELECT 'Value for Cost', Sat_Value FROM survey
    UNION ALL SELECT 'Signage', Sat_Signs FROM survey
    UNION ALL SELECT 'Parking', Sat_Parking FROM survey
    UNION ALL SELECT 'Admission Fee', Sat_Admission FROM survey
    UNION ALL SELECT 'Cleanliness', Sat_Cleanliness FROM survey
    UNION ALL SELECT 'Exhibit Variety', Sat_Variety FROM survey
    UNION ALL SELECT 'Map', Sat_Map FROM survey
    UNION ALL SELECT 'Employee Helpfulness', Sat_Emp_Helpfulness FROM survey
    UNION ALL SELECT 'Volunteer Helpfulness', Sat_Vol_Helpfulness FROM survey
    UNION ALL SELECT 'Gift Shops', Sat_Gift_Shops FROM survey
    UNION ALL SELECT 'Accessibility', Sat_Accessibility FROM survey
    UNION ALL SELECT 'Green Policies', Sat_Green FROM survey
),
agg AS (
    SELECT
        feature,
        COUNT(sat) AS n_rated,
        ROUND(100.0 * COUNT(sat) / 7522, 1) AS pct_rated,
        ROUND(AVG(sat), 2) AS avg_satisfaction,
        ROUND(100.0 * SUM(CASE WHEN sat >= 4 THEN 1 ELSE 0 END) / COUNT(sat), 1) AS pct_satisfied
    FROM features
    WHERE sat IS NOT NULL
    GROUP BY feature
)
SELECT
    a.*,
    CASE 
        WHEN pct_rated > (SELECT AVG(pct_rated) FROM agg)
         AND pct_satisfied < (SELECT AVG(pct_satisfied) FROM agg)
        THEN 'PAIN POINT'
        WHEN pct_rated <= (SELECT AVG(pct_rated) FROM agg)
         AND pct_satisfied >= (SELECT AVG(pct_satisfied) FROM agg)
        THEN 'Hidden Gem'
        ELSE 'Monitor'
    END AS quadrant,
    RANK() OVER (ORDER BY pct_satisfied) AS sat_rank
FROM agg a
ORDER BY sat_rank
""")

,feature,n_rated,pct_rated,avg_satisfaction,pct_satisfied,quadrant,sat_rank
0,Food Services,4771,63.4,3.25,40.1,Monitor,1
1,Value for Cost,7468,99.3,3.54,55.7,PAIN POINT,2
2,Gift Shops,3913,52.0,3.68,56.6,Monitor,3
3,Washrooms,6963,92.6,3.64,57.5,PAIN POINT,4
4,Parking,6938,92.2,3.69,59.9,PAIN POINT,5
5,Admission Fee,7268,96.6,3.74,61.1,PAIN POINT,6
6,Signage,7451,99.1,3.77,64.1,PAIN POINT,7
7,Map,6725,89.4,3.87,67.8,Monitor,8
8,Accessibility,4627,61.5,3.90,69.2,Hidden Gem,9
9,Exhibit Variety,7449,99.0,3.93,71.7,Monitor,10


## 8. Complaint Theme Prevalence
How common is each complaint theme extracted from the topic model? What's the average satisfaction of visitors who raised each complaint?

In [104]:
sql("""
WITH themes AS (
    SELECT 'Animal Visibility' AS theme, theme_animal_visibility_availability_flag AS flag, Overall_Sat AS sat FROM survey
    UNION ALL SELECT 'Construction Closures', theme_construction_closures_disruption_flag, Overall_Sat FROM survey
    UNION ALL SELECT 'Amenities Comfort', theme_amenities_comfort_flag, Overall_Sat FROM survey
    UNION ALL SELECT 'Operational Reliability', theme_operational_reliability_flag, Overall_Sat FROM survey
    UNION ALL SELECT 'Mobility Accessibility', theme_mobility_accessibility_flag, Overall_Sat FROM survey
    UNION ALL SELECT 'Cost Value Pricing', theme_cost_value_pricing_flag, Overall_Sat FROM survey
    UNION ALL SELECT 'Navigation Info', theme_navigation_info_flag, Overall_Sat FROM survey
),
agg as (
    SELECT
        theme,
        SUM(flag) AS n_mentioned,
        ROUND(100.0 * SUM(flag) / 7522, 2) AS pct_mentioned,
        ROUND(AVG(CASE WHEN flag = 1 THEN sat END), 2) AS avg_satisfaction,
        ROUND(100.0 * SUM(CASE WHEN flag = 1 AND sat >= 4 THEN 1 ELSE 0 END) / NULLIF(SUM(flag), 0), 1) AS pct_satisfied
    FROM themes
    GROUP BY theme
)
SELECT
    a.*,
    CASE 
        WHEN pct_mentioned > (SELECT AVG(pct_mentioned) FROM agg)
         AND pct_satisfied < (SELECT AVG(pct_satisfied) FROM agg)
        THEN 'PAIN POINT'
        WHEN pct_mentioned <= (SELECT AVG(pct_mentioned) FROM agg)
         AND pct_satisfied >= (SELECT AVG(pct_satisfied) FROM agg)
        THEN 'Hidden Gem'
        ELSE 'Monitor'
    END AS quadrant,
    RANK() OVER (ORDER BY pct_satisfied) AS sat_rank
FROM agg a
ORDER BY sat_rank
""")

,theme,n_mentioned,pct_mentioned,avg_satisfaction,pct_satisfied,quadrant,sat_rank
0,Cost Value Pricing,162,2.15,3.01,43.2,Monitor,1
1,Animal Visibility,593,7.88,3.16,45.4,PAIN POINT,2
2,Construction Closures,482,6.41,3.31,48.5,PAIN POINT,3
3,Navigation Info,129,1.71,3.66,62.0,Hidden Gem,4
4,Operational Reliability,523,6.95,3.67,64.6,Monitor,5
5,Amenities Comfort,660,8.77,3.87,73.9,Monitor,6
6,Mobility Accessibility,373,4.96,3.99,75.6,Hidden Gem,7


In [105]:
conn.close()